Setup: run this first:

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, os

df = pd.read_parquet('/content/drive/MyDrive/cfpb/complaints_2025.parquet')

credit_reporting = 'Credit reporting or other personal consumer reports'
bureaus = ['EQUIFAX, INC.', 'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'Experian Information Solutions Inc.']
relief = ['Closed with monetary relief', 'Closed with non-monetary relief']

other = df[(df['Product'] != credit_reporting) &
           (df['Company response to consumer'] != 'In progress') &
           (~df['Company'].isin(bureaus))]

def scorecard(data, keys, min_n):
    s = data.groupby(keys, observed=True).agg(
        complaints=('Complaint ID', 'count'),
        timely_rate=('Timely response?', lambda x: (x == 'Yes').mean()),
        monetary_rate=('Company response to consumer', lambda x: (x == 'Closed with monetary relief').mean()),
        any_relief_rate=('Company response to consumer', lambda x: x.isin(relief).mean()))
    return s[s['complaints'] >= min_n].round(3)

summary = scorecard(other, 'Company', 1000).sort_values('complaints', ascending=False)
by_product = scorecard(other, ['Product', 'Company'], 500)

jan_firms = ['Block, Inc.', 'Early Warning Services, LLC',
             'NAVY FEDERAL CREDIT UNION', 'CAPITAL ONE FINANCIAL CORPORATION']
jan = other[(other['Company'].isin(jan_firms)) & (other['Date received'] < '2025-02-01')]

print(other.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(538171, 13)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.listdir('/content/drive/MyDrive/cfpb')

['CCDB_Export_9_January_2025_through_February_2025.csv',
 'CCDB_Export_10_March_2025_through_April_2025.zip',
 'CCDB_Export_11_May_2025_through_June_2025.zip',
 'CCDB_Export_12_July_2025_through_August_2025.zip']

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/cfpb/CCDB_Export_9_January_2025_through_February_2025.csv', low_memory=False)
df.shape

(811019, 16)

In [ ]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
       'Company response to consumer', 'Timely response?', 'Complaint ID'],
      dtype='object')

In [ ]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-04,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,PR,00949,NaN,Web,2025-01-04,Closed with explanation,Yes,11383417
1,2025-01-04,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",PR,00949,NaN,Web,2025-01-04,Closed with non-monetary relief,Yes,11383088
2,2025-01-04,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",PR,00949,NaN,Web,2025-01-04,Closed with non-monetary relief,Yes,11382832
3,2025-01-25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,34744,NaN,Web,2025-01-25,Closed with non-monetary relief,Yes,11725468
4,2025-01-25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,34744,NaN,Web,2025-01-25,Closed with explanation,Yes,11760322


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 811019 entries, 0 to 811018
Data columns (total 16 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 811019 non-null  object
 1   Product                       811019 non-null  object
 2   Sub-product                   811019 non-null  object
 3   Issue                         811019 non-null  object
 4   Sub-issue                     750518 non-null  object
 5   Consumer complaint narrative  265405 non-null  object
 6   Company public response       459070 non-null  object
 7   Company                       811019 non-null  object
 8   State                         809290 non-null  object
 9   ZIP code                      811019 non-null  object
 10  Tags                          31107 non-null   object
 11  Submitted via                 811019 non-null  object
 12  Date sent to company          811019 non-null  object
 13 

In [ ]:
df.isna().mean().round(3)

,0
Date received,0.000
Product,0.000
Sub-product,0.000
Issue,0.000
Sub-issue,0.075
Consumer complaint narrative,0.673
Company public response,0.434
Company,0.000
State,0.002
ZIP code,0.000


## Data Overview

The initial file contains **811,019 complaints** across **16 columns**.

**Data completeness:**
- Core fields used in the main analysis (date received, product, issue, company, company response, and timely response) are essentially complete, with no meaningful missing values.
- **Consumer complaint narrative** is missing for 67% of complaints, since consumers choose whether to share a written description. The roughly one-third with narratives is sufficient for an optional text analysis of complaints involving automated systems.
- **Company public response** is missing for 43% of complaints, since companies may choose whether to post a public response.
- **Tags** are missing for 96% of complaints. Tags identify special populations, such as older Americans and servicemembers, and will be used only for targeted analysis.
- **Sub-issue** (7.5% missing) and **state** (0.2% missing) have minor gaps that are unlikely to affect results.

**Decision:** For the main analysis, I exclude the narrative and ZIP code columns to reduce memory use, since state-level geography is sufficient. Narratives will be revisited separately for the text analysis extension.

In [ ]:
import glob

cols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
        'Company', 'State', 'Tags', 'Submitted via', 'Date sent to company',
        'Company response to consumer', 'Timely response?', 'Complaint ID']

files = glob.glob('/content/drive/MyDrive/cfpb/*')
df = pd.concat([pd.read_csv(f, usecols=cols, low_memory=False) for f in files])

df['Date received'] = pd.to_datetime(df['Date received'])
df = df.drop_duplicates(subset='Complaint ID')
df = df[df['Date received'].dt.year == 2025]

for c in ['Product', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State',
          'Tags', 'Submitted via', 'Company response to consumer', 'Timely response?']:
    df[c] = df[c].astype('category')

df.shape

(5442964, 13)

In [ ]:
df.to_parquet('/content/drive/MyDrive/cfpb/complaints_2025.parquet')

In [ ]:
df['Product'].value_counts()

,count
Product,
Credit reporting or other personal consumer reports,4810302
Debt collection,283204
Credit card,89978
"Money transfer, virtual currency, or money service",84619
Checking or savings account,84195
Mortgage,24697
Student loan,21295
Vehicle loan or lease,20769
"Payday loan, title loan, personal loan, or advance loan",13137


In [ ]:
df['Company'].value_counts().head(20)

,count
Company,
"EQUIFAX, INC.",1680536
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",1601849
Experian Information Solutions Inc.,1388821
"Block, Inc.",48091
CAPITAL ONE FINANCIAL CORPORATION,34102
"CBC Companies, Inc.",30769
"Early Warning Services, LLC",24509
JPMORGAN CHASE & CO.,24458
Resurgent Capital Services L.P.,23741


In [ ]:
df['Company response to consumer'].value_counts()

,count
Company response to consumer,
Closed with explanation,3194726
Closed with non-monetary relief,2212234
Closed with monetary relief,26125
Untimely response,9594
In progress,285


In [ ]:
df['Timely response?'].value_counts()

,count
Timely response?,
Yes,5418415
No,24549


## Initial Findings and Analytical Approach

The combined 2025 dataset contains **5,442,964 complaints**.

**Credit reporting dominates complaint volume.** Complaints about credit reporting account for 4,810,302 complaints, or 88% of the total. The three national credit bureaus (Equifax, TransUnion, and Experian) alone account for roughly 4.67 million complaints, about 86% of all 2025 complaints. The remaining ~633,000 complaints are spread across all other products, led by debt collection (about 45% of non-credit-reporting complaints), followed by credit cards, money transfers, and checking or savings accounts.

**Monetary relief is rare.** Across all complaints, 59% were closed with an explanation, 41% with non-monetary relief, and only 0.5% with monetary relief.

**Timely responses are nearly universal.** Over 99.5% of company responses were timely, which makes the small number of late responders notable.

**Decision:** Because credit reporting complaints are so concentrated among three companies, analyzing all complaints together would obscure patterns among banks, fintechs, and debt collectors. I therefore analyze two groups separately:
1. **Credit reporting complaints**
2. **All other products**

I also exclude the 285 complaints still marked "In progress" from response-quality measures, since they do not yet have a final outcome.

In [ ]:
credit_reporting = 'Credit reporting or other personal consumer reports'

cr = df[df['Product'] == credit_reporting]
other = df[(df['Product'] != credit_reporting) & (df['Company response to consumer'] != 'In progress')]

print(cr.shape, other.shape)

(4810302, 13) (632442, 13)


In [ ]:
other['Product'].value_counts()
other['Company'].value_counts().head(20)
other['Issue'].value_counts().head(15)

,count
Issue,
Attempts to collect debt not owed,115679
Written notification about debt,64048
Other transaction problem,53892
Took or threatened to take negative or legal action,48567
Managing an account,44959
False statements or representation,38455
Problem with a purchase shown on your statement,19810
Incorrect information on your report,19448
Trouble during payment process,12680


In [ ]:
display(other['Product'].value_counts())
display(other['Company'].value_counts().head(20))

,count
Product,
Debt collection,283183
Credit card,89977
"Money transfer, virtual currency, or money service",84617
Checking or savings account,84194
Mortgage,24697
Student loan,21101
Vehicle loan or lease,20769
"Payday loan, title loan, personal loan, or advance loan",13137
Prepaid card,6502


,count
Company,
"Block, Inc.",47897
"EQUIFAX, INC.",35561
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",31776
Experian Information Solutions Inc.,26934
CAPITAL ONE FINANCIAL CORPORATION,22971
"Early Warning Services, LLC",20570
JPMORGAN CHASE & CO.,19736
Resurgent Capital Services L.P.,17938
WELLS FARGO & COMPANY,15899


In [ ]:
bureaus = ['EQUIFAX, INC.', 'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'Experian Information Solutions Inc.']
other[other['Company'].isin(bureaus)]['Product'].value_counts().head(5)

,count
Product,
Debt collection,70641
Credit card,21205
Debt or credit management,750
Vehicle loan or lease,544
Checking or savings account,448


In [ ]:
summary = other.groupby('Company', observed=True).agg(
    complaints=('Complaint ID', 'count'),
    timely_rate=('Timely response?', lambda s: (s == 'Yes').mean()),
    monetary_rate=('Company response to consumer', lambda s: (s == 'Closed with monetary relief').mean()),
    any_relief_rate=('Company response to consumer', lambda s: s.isin(['Closed with monetary relief', 'Closed with non-monetary relief']).mean())
)

summary = summary[summary['complaints'] >= 1000].sort_values('complaints', ascending=False)
summary.round(3).head(25)

,complaints,timely_rate,monetary_rate,any_relief_rate
Company,,,,
"Block, Inc.",47897,0.982,0.001,0.001
"EQUIFAX, INC.",35561,1.000,0.000,0.555
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",31776,1.000,0.000,0.708
Experian Information Solutions Inc.,26934,1.000,0.001,0.004
CAPITAL ONE FINANCIAL CORPORATION,22971,1.000,0.051,0.145
"Early Warning Services, LLC",20570,1.000,0.000,0.000
JPMORGAN CHASE & CO.,19736,1.000,0.075,0.131
Resurgent Capital Services L.P.,17938,1.000,0.000,0.308
WELLS FARGO & COMPANY,15899,1.000,0.101,0.189


## Non-Credit-Reporting Complaints: Overview

After excluding credit reporting, 632,442 complaints remain. **Debt collection** accounts for about 45%, followed by **credit cards** (14%), **money transfers and payment apps** (13%), and **checking or savings accounts** (13%).

The most common issues are concentrated in debt collection, including attempts to collect debt not owed (115,679 complaints), written notification about debt, threatened legal action, and false statements. Many of these correspond to conduct regulated under the Fair Debt Collection Practices Act (FDCPA).

**Block, Inc. (Cash App)** receives the most complaints of any company in this group, and four payment fintechs (Block, Early Warning Services/Zelle, Chime, and PayPal) appear among the top 20 companies.

**Note on comparisons:** Raw complaint counts largely reflect company size. To compare companies fairly, the next section evaluates complaint-handling quality (timely response, monetary relief, and any relief rates) for companies with at least 1,000 complaints.

In [ ]:
other = other[~other['Company'].isin(bureaus)]
other.shape

(538171, 13)

In [ ]:
other['Product'].value_counts()

,count
Product,
Debt collection,212542
"Money transfer, virtual currency, or money service",84582
Checking or savings account,83746
Credit card,68772
Mortgage,24656
Student loan,20790
Vehicle loan or lease,20225
"Payday loan, title loan, personal loan, or advance loan",12986
Prepaid card,6357


## Data Quality Check: Credit Bureau Complaints Outside Credit Reporting

Even after excluding the credit reporting product, the three national credit bureaus (Equifax, TransUnion, and Experian) account for about 94,000 complaints in the remaining data, mainly filed under **debt collection** (70,641) and **credit card** (21,205).

Because the bureaus do not collect debts or issue credit cards, these complaints most likely reflect consumers disputing collection or credit card accounts appearing on their credit reports. In effect, they function as credit reporting complaints filed under a different product label. Left in, they would account for roughly 25% of debt collection complaints and overstate complaints against debt collectors.

**Decision:** I remove the three credit bureaus from the non-credit-reporting group so that the analysis of debt collection, banking, and fintech complaints reflects the companies actually providing those products.

After removing the credit bureaus, 538,171 complaints remain. Debt collection still leads (39.5%), but money transfers and payment services (15.7%) and checking or savings accounts (15.6%) now rank second and third, ahead of credit cards (12.8%). Payment and banking products together account for about one-third of complaints in this group.

In [ ]:
summary = other.groupby('Company', observed=True).agg(
    complaints=('Complaint ID', 'count'),
    timely_rate=('Timely response?', lambda s: (s == 'Yes').mean()),
    monetary_rate=('Company response to consumer', lambda s: (s == 'Closed with monetary relief').mean()),
    any_relief_rate=('Company response to consumer', lambda s: s.isin(['Closed with monetary relief', 'Closed with non-monetary relief']).mean())
)

summary = summary[summary['complaints'] >= 1000].sort_values('complaints', ascending=False)
summary.round(3).head(25)

,complaints,timely_rate,monetary_rate,any_relief_rate
Company,,,,
"Block, Inc.",47897,0.982,0.001,0.001
CAPITAL ONE FINANCIAL CORPORATION,22971,1.000,0.051,0.145
"Early Warning Services, LLC",20570,1.000,0.000,0.000
JPMORGAN CHASE & CO.,19736,1.000,0.075,0.131
Resurgent Capital Services L.P.,17938,1.000,0.000,0.308
WELLS FARGO & COMPANY,15899,1.000,0.101,0.189
"BANK OF AMERICA, NATIONAL ASSOCIATION",14948,0.996,0.292,0.336
"CITIBANK, N.A.",14831,1.000,0.231,0.498
NAVY FEDERAL CREDIT UNION,14191,1.000,0.049,0.154


## Company Scorecard: Complaint Handling Quality

For companies with at least 1,000 complaints, I calculated timely response rates, monetary relief rates, and any relief (monetary or non-monetary) rates.

**Key findings:**
- **Timeliness:** Nearly all companies respond on time over 99% of the time. **MOHELA** is a major outlier at 40%, followed by CCS Financial Services at 65%.
- **Variation among banks:** Citibank (50%) and Bank of America (34%) provided relief far more often than JPMorgan Chase (13%), Capital One (15%), and Navy Federal (15%).
- **Near-zero relief:** Block (Cash App) provided relief in just 0.1% of nearly 48,000 complaints. Early Warning Services (Zelle), CL Holdings, Nelnet, National Credit Systems, and Transworld Systems provided essentially none.
- **Debt buyers:** Resurgent, Encore, and Portfolio Recovery Associates provided no monetary relief but non-monetary relief in 31–45% of complaints, consistent with ending collection or correcting reporting.
- **Anomaly:** Kriya Capital reported non-monetary relief in 100% of its 4,219 complaints, a uniform pattern that warrants further investigation.

**Caveats:** Outcomes are self-reported by companies and not verified by the CFPB. Relief norms differ by product and business model, so low relief rates indicate areas for investigation rather than proof of misconduct.

In [ ]:
by_product = other.groupby(['Product', 'Company'], observed=True).agg(
    complaints=('Complaint ID', 'count'),
    timely_rate=('Timely response?', lambda s: (s == 'Yes').mean()),
    any_relief_rate=('Company response to consumer', lambda s: s.isin(['Closed with monetary relief', 'Closed with non-monetary relief']).mean())
)
by_product = by_product[by_product['complaints'] >= 500].round(3)

display(by_product.loc['Money transfer, virtual currency, or money service'].sort_values('complaints', ascending=False))
display(by_product.loc['Checking or savings account'].sort_values('complaints', ascending=False))

,complaints,timely_rate,any_relief_rate
Company,,,
"Block, Inc.",42569,0.988,0.000
"Early Warning Services, LLC",19703,1.000,0.000
"Paypal Holdings, Inc",5482,1.000,0.303
JPMORGAN CHASE & CO.,1967,1.000,0.041
Chime Financial Inc,1670,1.000,0.050
"BANK OF AMERICA, NATIONAL ASSOCIATION",1598,0.998,0.225
WELLS FARGO & COMPANY,1594,1.000,0.074
"Coinbase, Inc.",1294,0.988,0.264
ROBINHOOD MARKETS INC.,1086,0.867,0.000


,complaints,timely_rate,any_relief_rate
Company,,,
NAVY FEDERAL CREDIT UNION,10606,1.000,0.082
JPMORGAN CHASE & CO.,8937,1.000,0.115
WELLS FARGO & COMPANY,8181,1.000,0.175
CAPITAL ONE FINANCIAL CORPORATION,7912,1.000,0.039
"BANK OF AMERICA, NATIONAL ASSOCIATION",6680,0.996,0.401
Chime Financial Inc,5373,1.000,0.113
"Block, Inc.",3575,0.942,0.000
"CITIBANK, N.A.",3458,1.000,0.401
"SOFI TECHNOLOGIES, INC.",1968,1.000,0.080


## Within-Product Comparisons

**Money transfers and payment apps:** Block (Cash App) and Early Warning Services (Zelle) account for about 74% of complaints in this category, and both provided zero relief. By contrast, PayPal (30%) and Coinbase (26%) provided relief in a meaningful share of complaints, suggesting near-zero relief is not inherent to payment products. Robinhood had both the lowest timeliness (87%) and zero relief.

**Checking and savings accounts:** Relief rates vary by up to tenfold among banks offering the same product. Bank of America, Citibank, Huntington, TD Bank, BMO, Ally, and Citizens provided relief in 29–40% of complaints, while Capital One (3.9%), Navy Federal, SoFi, PNC, and Fifth Third provided relief in 4–9%. Block, Early Warning Services, FIS, Robinhood, Goldman Sachs, and Netspend provided essentially none. Navy Federal received the most checking complaints of any company but one of the lowest relief rates.

**Interpretation:** Differences of this size may reflect real variation in complaint handling, inconsistent self-reporting of outcomes across companies, or both. Either way, the results suggest that self-reported outcome data are difficult to compare across institutions without standardized definitions.

In [ ]:
monthly = (other.groupby([pd.Grouper(key='Date received', freq='MS'), 'Company'], observed=True)
           .size().rename('complaints').reset_index())

top = other['Company'].value_counts().head(30).index
monthly = monthly[monthly['Company'].isin(top)]

stats = monthly.groupby('Company', observed=True)['complaints'].agg(['mean', 'std'])
monthly = monthly.join(stats, on='Company')
monthly['z'] = (monthly['complaints'] - monthly['mean']) / monthly['std']

monthly[monthly['z'] > 2].sort_values('z', ascending=False)

,Date received,Company,complaints,mean,std,z
1010,2025-01-01,NAVY FEDERAL CREDIT UNION,7930,1182.583333,2129.594304,3.168405
533,2025-01-01,"Early Warning Services, LLC",16732,1714.166667,4746.047865,3.164282
243,2025-01-01,"Block, Inc.",31270,3991.416667,8623.906829,3.163135
273,2025-01-01,CAPITAL ONE FINANCIAL CORPORATION,6322,1914.250000,1396.709973,3.155809
4046,2025-03-01,"Nelnet, Inc.",972,402.333333,194.265205,2.932417
17501,2025-12-01,"Bread Financial Holdings, Inc.",854,479.833333,138.248020,2.706488
17963,2025-12-01,GOLDMAN SACHS BANK USA,460,311.083333,55.671861,2.674900
18766,2025-12-01,TRUIST FINANCIAL CORPORATION,417,301.250000,44.942640,2.575505
14335,2025-10-01,Chime Financial Inc,1488,671.416667,328.854059,2.483118
18616,2025-12-01,"SANTANDER HOLDINGS USA, INC.",452,290.083333,65.738405,2.463045


In [ ]:
jan_firms = ['Block, Inc.', 'Early Warning Services, LLC',
             'NAVY FEDERAL CREDIT UNION', 'CAPITAL ONE FINANCIAL CORPORATION']

jan = other[(other['Company'].isin(jan_firms)) & (other['Date received'] < '2025-02-01')]
jan.groupby([jan['Date received'].dt.date, 'Company'], observed=True).size().unstack()

Company,"Block, Inc.",CAPITAL ONE FINANCIAL CORPORATION,"Early Warning Services, LLC",NAVY FEDERAL CREDIT UNION
Date received,,,,
2025-01-01,4,20,2,14
2025-01-02,20,42,6,17
2025-01-03,14,34,5,27
2025-01-04,12,42,5,19
2025-01-05,8,30,2,9
2025-01-06,17,39,8,18
2025-01-07,14,50,7,24
2025-01-08,11,42,4,34
2025-01-09,16,52,9,21


In [ ]:
other.groupby(pd.Grouper(key='Date received', freq='MS')).size()

,0
Date received,
2025-01-01,100747
2025-02-01,35642
2025-03-01,36291
2025-04-01,33272
2025-05-01,35781
2025-06-01,35828
2025-07-01,41007
2025-08-01,39175
2025-09-01,40343


## Anomaly Detection: Monthly Complaint Spikes

For the 30 companies with the most complaints, I calculated each month's z-score relative to that company's 2025 average to flag unusually high months.

**January 2025:** Four companies had extreme spikes, with a single month accounting for a large share of their annual complaints: Early Warning Services/Zelle (81%), Block/Cash App (65%), Navy Federal (56%), and Capital One (28%). These spikes coincide with major CFPB enforcement actions announced between December 2024 and January 2025 involving each company. This suggests that high-profile enforcement publicity may drive complaint volume, though timing alone does not establish causation. Several of these actions were later dismissed or withdrawn in 2025.

**Methodological note:** With 12 monthly observations, the maximum possible z-score is about 3.18. The four January spikes reach this ceiling, indicating that one month dominates each company's annual total. Because extreme months also inflate the mean and standard deviation, z-scores understate the true size of these spikes.

**December 2025:** Many companies across product types were modestly elevated, which suggests a market-wide pattern rather than company-specific issues. This is examined further below.

In [ ]:
narr = pd.read_csv('/content/drive/MyDrive/cfpb/CCDB_Export_9_January_2025_through_February_2025.csv',
                   usecols=['Date received', 'Company', 'Consumer complaint narrative'],
                   low_memory=False)

narr = narr[narr['Company'].isin(['Block, Inc.', 'Early Warning Services, LLC'])]
narr = narr.dropna(subset=['Consumer complaint narrative'])

text = narr['Consumer complaint narrative'].str.lower().str.strip()
print(len(text), 'narratives;', round(text.duplicated(keep=False).mean(), 3), 'share appearing more than once')
text.value_counts().head(5)

44880 narratives; 0.553 share appearing more than once


,count
Consumer complaint narrative,
"i am filing a complaint against cash app ( block , inc. ) due to inadequate customer service and unfair practices, which violate the consumer financial protection act ( cfpa ). specifically, cash app failed to take timely and effective measures to prevent and address fraud on their platform, leaving my account vulnerable and unprotected.\n\nfurthermore, their dispute resolution process was unfair and deceptive, as they did not comply with error resolution requirements under the electronic fund transfer act ( efta ) and regulation e. these actions have caused significant inconvenience and financial loss, and i seek appropriate redress for these violations.",7526
"i am writing to formally express my concerns regarding zelle 's handling of disputes, particularly in light of the recent cfpb lawsuit highlighting significant issues with the platform. despite using zelle numerous times and reporting discrepancies, my financial institution and zelle failed to investigate my claims adequately. they informed me that recovering lost funds was impossible and that tracking these transactions was unfeasible. the cfpb 's findings indicate that zelle has not only neglected to protect consumers but has also allowed repeat offenders to exploit the system, leaving users vulnerable to scams. this lack of accountability and transparency is unacceptable and raises serious concerns about consumer safety.",3614
"i am filing a complaint against cash app ( block , inc. ) due to inadequate customer service and unfair practices, which violate the consumer financial protection act ( cfpa ). specifically, cash app failed to take timely and effective measures to prevent and address fraud on their platform, leaving my account vulnerable and unprotected. furthermore, their dispute resolution process was unfair and deceptive, as they did not comply with error resolution requirements under the electronic fund transfer act ( efta ) and regulation e. these actions have caused significant inconvenience and financial loss, and i seek appropriate redress for these violations.",1507
"i am filing a complaint against cash app ( block , inc. ) due to inadequate customer service and unfair practices, which violate the consumer financial protection act ( cfpa ). specifically, cash app failed to take timely and effective measures to prevent and address fraud on their platform, leaving my account vulnerable and unprotected.\n\nfurthermore, their dispute resolution process was unfair and deceptive, as they did not comply with error resolution requirements under the electronic fund transfer act ( efta ) and regulation e. these actions have caused significant inconvenience and financial loss, and i seek appropriate redress for these violations",1035
"i am filing a complaint against cash app ( block , inc. ) due to inadequate customer service and unfair practices, which violate the consumer financial protection act ( cfpa ). specifically, cash app failed to take timely and effective measures to prevent and address fraud on their platform, leaving my account vulnerable and unprotected.\n\nfurthermore, their dispute resolution proce was unfair and deceptive, as they did not comply with error resolution requirements under the electronic fund transfer act ( efta ) and regulation e. these actions have caused significant inconvenience and financial loss, and i seek appropriate redress for these violations.",814


In [2]:
feb_on = other[other['Date received'] >= '2025-02-01']

feb_on[feb_on['Company'].isin(jan_firms)].groupby('Company', observed=True)[
    'Company response to consumer'].apply(lambda s: s.isin(relief).mean()).round(3)

,Company response to consumer
Company,
"Block, Inc.",0.002
CAPITAL ONE FINANCIAL CORPORATION,0.182
"Early Warning Services, LLC",0.000
NAVY FEDERAL CREDIT UNION,0.301


In [3]:
out = '/content/drive/MyDrive/cfpb/tableau/'
os.makedirs(out, exist_ok=True)

(other.groupby([pd.Grouper(key='Date received', freq='MS'), 'Product'], observed=True)
 .size().rename('complaints').reset_index()
 .to_csv(out + 'monthly_by_product.csv', index=False))

summary.reset_index().to_csv(out + 'company_scorecard.csv', index=False)
by_product.reset_index().to_csv(out + 'scorecard_by_product.csv', index=False)

(jan.groupby([jan['Date received'].dt.date, 'Company'], observed=True)
 .size().rename('complaints').reset_index()
 .to_csv(out + 'january_daily.csv', index=False))

print(os.listdir(out))

['monthly_by_product.csv', 'company_scorecard.csv', 'scorecard_by_product.csv', 'january_daily.csv']


## Robustness Check: Relief Rates Excluding January

To test whether the January spike distorted relief rates, I recalculated relief rates for the four affected companies using only complaints from February onward.

- **Block (0.2%) and Early Warning Services (0.0%)** still provided essentially no relief, so their near-zero relief rates are not solely an artifact of the January surge.
- **Navy Federal's** relief rate doubled from 15.4% to 30.1%. Its January complaints received relief only about 4% of the time, so its low full-year rate was driven by the January wave rather than its typical complaint handling.
- **Capital One's** rate rose modestly from 14.5% to 18.2%, remaining on the lower end among large banks.

**Takeaway:** Short-term complaint surges can significantly distort company-level metrics, so year-level comparisons should be checked against periods without unusual events.